# pipe_catedra/04 — Entrenamiento final y submit (porteo de `final_traing_4.ipynb` / z304)

Input  : `z302_features_{modo}.parquet`, `z302_inferencia_{modo}.parquet`, `z303_hiper_{modo}_{experimento_hiper}.json`
Output : `z304_predicciones_{modo}.csv` -> Kaggle

Porteo fiel del notebook de la catedra -- misma logica, mismas palancas.

Responsabilidades:
- Leer hiperparametros del disco (sin re-correr Optuna)
- Entrenar LGBM con features categoricas declaradas
- Predecir sobre inferencia (201912 -> 202002)
- Submit a Kaggle


## 0) Setup


In [ ]:
import os, shutil, subprocess
from pathlib import Path

import polars as pl
import numpy as np
import pandas as pd
import lightgbm as lgb
import json


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_OUT = BUCKET / "exp_pipe_catedra"
DIR_OUT.mkdir(parents=True, exist_ok=True)
print(f"BUCKET: {BUCKET}")
print(f"salida: {DIR_OUT}")

# Kaggle auth: usar el de ~/.kaggle si ya existe; si no, buscarlo en el bucket
kaggle_dst = Path.home() / ".kaggle" / "kaggle.json"
kaggle_dst.parent.mkdir(parents=True, exist_ok=True)
if kaggle_dst.exists():
    kaggle_dst.chmod(0o600)
    print("Kaggle auth OK (ya estaba en ~/.kaggle)")
else:
    _encontrado = False
    for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
        if cand.exists():
            shutil.copy(cand, kaggle_dst)
            kaggle_dst.chmod(0o600)
            print(f"Kaggle auth OK (copiado de {cand})")
            _encontrado = True
            break
    if not _encontrado:
        print("aviso: kaggle.json no encontrado. Subilo a ~/.kaggle/kaggle.json o al bucket.")


## 1) Parametros — palancas


In [ ]:
PARAM = {
    'experimento': 'z304',
    'kaggle_competition': 'labo-iii-2026-rosario',

    # PALANCA 1 (debe coincidir con 01_/02_/03_)
    'modo_agrupacion': 'producto',

    'path_apredecir': str(DIR_RAW / 'product_id_apredecir201912.txt'),

    # PALANCA 12: sampling de filas para entrenamiento
    'sampling_frac': None,

    # PALANCA 14: clip de negativos
    'clip_min': 0.0,

    # PALANCA 16: peso por recencia (debe coincidir con 03_)
    'decay_recencia': None,

    # PALANCA 17: que experimento de hiperparametros usar
    # nombre del 'experimento' de 03_ cuyo JSON se quiere cargar
    'experimento_hiper': 'z303',

    # PALANCA 18: ensemble de semillas
    # lista de semillas: entrena un modelo por cada una y promedia las predicciones.
    # [102191] = un solo modelo (sin ensemble)
    'semillas_ensemble': [102191],

    'semilla': 102191,

    'submit': False,
}

MODO = PARAM['modo_agrupacion']
PARAM['path_train']  = str(DIR_OUT / f"z302_features_{MODO}.parquet")
PARAM['path_infer']  = str(DIR_OUT / f"z302_inferencia_{MODO}.parquet")
PARAM['path_hiper']  = str(DIR_OUT / f"z303_hiper_{MODO}_{PARAM['experimento_hiper']}.json")
PARAM['path_output'] = str(DIR_OUT / f"z304_predicciones_{MODO}.csv")

print('Parametros:', PARAM)
print('Train:', PARAM['path_train'])
print('Hiper:', PARAM['path_hiper'])


## 2) Cargar hiperparametros y datasets


In [ ]:
with open(PARAM['path_hiper']) as f:
    cfg = json.load(f)

FEATURES     = cfg['features']
CAT_FEATURES = cfg.get('cat_features', [])
TIPO_TARGET  = cfg.get('tipo_target', 'nivel')
TARGET_COL   = 'target_delta' if TIPO_TARGET == 'delta' else 'target_nivel'
METODO_ESCALADO = cfg.get('metodo_escalado')
hiper        = cfg['hiperparametros']

print(f'Experimento Optuna: {cfg["experimento"]}')
print(f'Metrica: {cfg["metrica"]} = {cfg["mejor_valor"]:.4f}')
print(f'Tipo de target: {TIPO_TARGET}')
print(f'Metodo de escalado: {METODO_ESCALADO}')
print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'Categoricas: {CAT_FEATURES}')

df_train = pl.read_parquet(PARAM['path_train'])
df_infer = pl.read_parquet(PARAM['path_infer'])
tb_pred  = pl.read_csv(PARAM['path_apredecir'], separator='\t',
                       schema_overrides={'product_id': pl.Int32})

# Modo de agrupacion propagado desde 01_ (define si hay que sumar por producto)
MODO = df_infer['modo_agrupacion'][0] if 'modo_agrupacion' in df_infer.columns else 'producto'
print(f'Modo de agrupacion: {MODO}')

print(f'\nTrain: {df_train.shape}  |  Inferencia: {df_infer.shape}')


In [ ]:
def escalar_target(y_raw, media, std, minr, maxr, metodo, es_delta):
    """Escala el target (nivel o delta) segun 'metodo_escalado' guardado por 03_Optuna.
    Los 4 parametros vienen de columnas calculadas en 02_FE (shift(1) -> sin leakage)."""
    if metodo is None:
        return y_raw
    if metodo == 'media':
        denom = np.where(media > 0, media, 1.0)
        return y_raw / denom
    elif metodo == 'zscore':
        denom = np.where(std > 0, std, 1.0)
        return y_raw / denom if es_delta else (y_raw - media) / denom
    elif metodo == 'rango':
        denom = np.where((maxr - minr) > 0, maxr - minr, 1.0)
        return y_raw / denom if es_delta else (y_raw - minr) / denom
    raise ValueError(f'metodo_escalado invalido: {metodo}')


def desescalar_target(y_scaled, media, std, minr, maxr, metodo, es_delta):
    """Inversa exacta de escalar_target."""
    if metodo is None:
        return y_scaled
    if metodo == 'media':
        denom = np.where(media > 0, media, 1.0)
        return y_scaled * denom
    elif metodo == 'zscore':
        denom = np.where(std > 0, std, 1.0)
        return y_scaled * denom if es_delta else y_scaled * denom + media
    elif metodo == 'rango':
        denom = np.where((maxr - minr) > 0, maxr - minr, 1.0)
        return y_scaled * denom if es_delta else y_scaled * denom + minr
    raise ValueError(f'metodo_escalado invalido: {metodo}')


## 3) Preparacion X / y (con categoricas)


In [ ]:
df_train_pd = df_train.to_pandas()
df_infer_pd = df_infer.to_pandas()

if PARAM['sampling_frac'] is not None:
    df_train_pd = df_train_pd.sample(frac=PARAM['sampling_frac'], random_state=PARAM['semilla'])
    print(f'Sampling: {len(df_train_pd):,} filas')

faltantes = [f for f in FEATURES if f not in df_train_pd.columns]
if faltantes:
    print(f'aviso: Features faltantes: {faltantes}')
    FEATURES = [f for f in FEATURES if f in df_train_pd.columns]

for c in CAT_FEATURES:
    if c in df_train_pd.columns:
        df_train_pd[c] = df_train_pd[c].astype('category')
        df_infer_pd[c] = df_infer_pd[c].astype('category')

X_train = df_train_pd[FEATURES]
y_train_raw = df_train_pd[TARGET_COL].values
y_train = escalar_target(
    y_train_raw,
    df_train_pd['media_rolling'].values, df_train_pd['std_rolling'].values,
    df_train_pd['min_rolling'].values, df_train_pd['max_rolling'].values,
    METODO_ESCALADO, TIPO_TARGET == 'delta'
)
X_infer = df_infer_pd[FEATURES]

print(f'X_train: {X_train.shape}  |  X_infer: {X_infer.shape}')
if METODO_ESCALADO is not None:
    print(f'Target escalado con metodo_escalado={METODO_ESCALADO!r} (heredado de 03_Optuna).')


## 4) Entrenamiento final


In [ ]:
OBJECTIVE_LGBM = cfg.get('objective_lgbm', 'regression')

def calcular_pesos(periodos_serie, decay):
    """Peso por recencia: el periodo mas reciente pesa 1, cada mes hacia atras decae."""
    if decay is None:
        return None
    periodos = sorted(periodos_serie.unique())
    idx = {p: i for i, p in enumerate(periodos)}
    n = len(periodos)
    return periodos_serie.map(lambda p: decay ** (n - 1 - idx[p])).values

w_train = calcular_pesos(df_train_pd['periodo'], PARAM['decay_recencia'])
if w_train is not None:
    print(f'Sample weights por recencia (decay={PARAM["decay_recencia"]})')

semillas = PARAM['semillas_ensemble']
print(f'Objective: {OBJECTIVE_LGBM}  |  Semillas: {semillas}')

modelos = []
for s in semillas:
    params_lgbm = {
        'objective':     OBJECTIVE_LGBM,
        'metric':        'mae',
        'verbosity':     -1,
        'boosting_type': 'gbdt',
        'seed':          s,
        **hiper
    }
    m = lgb.LGBMRegressor(**params_lgbm)
    m.fit(X_train, y_train, sample_weight=w_train, categorical_feature=CAT_FEATURES)
    modelos.append(m)
    print(f'  modelo semilla {s} entrenado (n_estimators={m.n_estimators_})')

print(f'Ensemble de {len(modelos)} modelo(s) listo.')


## 5) Feature importance


In [ ]:
fi = pd.Series(modelos[0].feature_importances_, index=FEATURES).sort_values(ascending=False)
print('Feature importance (gain, primer modelo del ensemble):')
print(fi.to_string())


## 6) Prediccion


In [ ]:
preds = np.column_stack([m.predict(X_infer) for m in modelos])
y_pred_scaled = preds.mean(axis=1)

y_pred = desescalar_target(
    y_pred_scaled,
    df_infer_pd['media_rolling'].values, df_infer_pd['std_rolling'].values,
    df_infer_pd['min_rolling'].values, df_infer_pd['max_rolling'].values,
    METODO_ESCALADO, TIPO_TARGET == 'delta'
)

if TIPO_TARGET == 'delta':
    tn_actual = df_infer_pd['tn'].values
    y_pred = tn_actual + y_pred
    print('Target delta: nivel reconstruido = tn_actual + delta')
else:
    print('Target nivel: prediccion directa')

y_pred = np.maximum(y_pred, PARAM['clip_min'])

print(f'min={y_pred.min():.3f}  max={y_pred.max():.3f}  mean={y_pred.mean():.3f}')
print(f'Predicciones = 0: {(y_pred == 0).sum()}')


## 7) Armar tabla de submit


In [ ]:
df_pred = pl.DataFrame({
    'product_id': df_infer_pd['product_id'].values.astype('int32'),
    'periodo':    df_infer_pd['periodo'].values.astype('int32'),
    'tn':         y_pred.astype('float64'),
})

ultimo_p = int(df_infer_pd['periodo'].max())
df_pred_p = df_pred.filter(pl.col('periodo') == ultimo_p)

# Kaggle mide a nivel product_id. Si el modo es cliente_producto,
# sumar las predicciones de todos los clientes de cada producto.
if MODO == 'cliente_producto':
    df_pred_final = (
        df_pred_p
        .group_by('product_id')
        .agg(pl.col('tn').sum())
    )
    print(f'Modo cliente_producto: agregado a nivel producto.')
else:
    df_pred_final = df_pred_p.drop('periodo')

print(f'Periodo usado: {ultimo_p}  |  Productos: {df_pred_final.height}')

# Completar con 0 los productos sin fila
tb_base = tb_pred.with_columns(pl.lit(0.0).alias('tn'))
tb_submit = (
    tb_base
    .join(df_pred_final, on='product_id', how='left', suffix='_pred')
    .with_columns(pl.coalesce(['tn_pred', 'tn']).alias('tn'))
    .select(['product_id', 'tn'])
    .sort('product_id')
)
print(f'Submit: {tb_submit.height} productos')
print(tb_submit.describe())


## 8) Guardar y submit


In [ ]:
tb_submit.write_csv(PARAM['path_output'])
print(f'Guardado: {PARAM["path_output"]}')

if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube. El CSV ya esta generado.")
else:
    mensaje = f"LGBM z304 | {cfg['metrica']}={cfg['mejor_valor']:.4f}"
    res = subprocess.run(
        ['kaggle', 'competitions', 'submit',
         '-c', PARAM['kaggle_competition'],
         '-f', PARAM['path_output'],
         '-m', mensaje],
        capture_output=True, text=True
    )
    print('stdout:', res.stdout)
    print('stderr:', res.stderr)
    print('returncode:', res.returncode)


## 9) Diagnostico (WAPE in-sample, orientativo)


In [ ]:
preds_tr = np.column_stack([m.predict(X_train) for m in modelos])
y_train_pred_scaled = preds_tr.mean(axis=1)
y_train_pred = desescalar_target(
    y_train_pred_scaled,
    df_train_pd['media_rolling'].values, df_train_pd['std_rolling'].values,
    df_train_pd['min_rolling'].values, df_train_pd['max_rolling'].values,
    METODO_ESCALADO, TIPO_TARGET == 'delta'
)

if TIPO_TARGET == 'delta':
    tn_tr      = df_train_pd['tn'].values
    pred_nivel = np.maximum(tn_tr + y_train_pred, 0.0)
    real_nivel = tn_tr + y_train_raw
else:
    pred_nivel = np.maximum(y_train_pred, 0.0)
    real_nivel = y_train_raw

wape_train = np.abs(real_nivel - pred_nivel).sum() / real_nivel.sum()
print(f'WAPE in-sample sobre nivel (NO es el de Kaggle): {wape_train:.4f}  ({wape_train*100:.2f}%)')
